# Stage 3 — Pose Estimation Evaluation

Run RTMPose-m top-down pose estimation on tracked persons.

**What this notebook covers:**
- Loading tracks from cache (or running tracking inline)
- Running `pipeline.pose.PoseEstimator` (RTMPose-m ONNX, top-down)
- Visualising 17-keypoint skeletons overlaid on frame grid
- Keypoint confidence histograms
- Saving poses to cache for metric extractors

**Note:** Requires `onnxruntime-gpu` or `onnxruntime` for CPU fallback.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

VIDEO_PATH = Path('../data/raw_footage/sample_clip.mp4')
TARGET_FPS = 15
MAX_FRAMES = 30    # pose is slow on CPU — keep small for quick eval
JOB_ID     = 'notebook-eval-detection'

from pipeline.cache import PipelineCache
cache = PipelineCache(job_id=JOB_ID, cache_root=Path('../data/cache'))
print("Stages cached:", cache.summary().get('stages_cached', []))

In [ ]:
from pipeline.ingest import extract_frames

frames, frame_indices, timestamps = [], [], []
for fi, frame, ts in extract_frames(str(VIDEO_PATH), target_fps=TARGET_FPS):
    frames.append(frame); frame_indices.append(fi); timestamps.append(ts)
    if MAX_FRAMES and len(frames) >= MAX_FRAMES: break

# Load or compute tracks
if cache.has('track'):
    all_tracks = cache.load_tracks()[:len(frames)]
    print(f"Loaded tracks from cache.")
else:
    from pipeline.detect import PersonDetector
    from pipeline.track import PersonTracker
    detector = PersonDetector()
    tracker  = PersonTracker()
    all_tracks = []
    all_dets   = []
    for i, frame in enumerate(frames):
        dets = detector.detect(frame)
        for d in dets: d.frame_idx = frame_indices[i]
        tracks = tracker.update(dets, frame_idx=frame_indices[i])
        all_dets.append(dets); all_tracks.append(tracks)
    cache.save_detections(all_dets); cache.save_tracks(all_tracks)
    print("Detection + tracking done.")

In [ ]:
from pipeline.pose import PoseEstimator

estimator = PoseEstimator()
all_poses = []

for i, (frame, tracks) in enumerate(zip(frames, all_tracks)):
    poses = estimator.estimate_batch(frame, tracks)
    all_poses.append(poses)
    if i % 5 == 0:
        print(f"  frame {frame_indices[i]:4d}  t={timestamps[i]:.2f}s  "
              f"tracks={len(tracks)}  poses={len(poses)}")

cache.save_poses(all_poses)
print("\nPose estimation complete.")

In [ ]:
# ── COCO-17 skeleton connections ─────────────────────────────────────────────
SKELETON = [
    (0, 1), (0, 2), (1, 3), (2, 4),         # face
    (5, 6),                                   # shoulders
    (5, 7), (7, 9), (6, 8), (8, 10),         # arms
    (5, 11), (6, 12), (11, 12),              # torso
    (11, 13), (13, 15), (12, 14), (14, 16),  # legs
]
PALETTE_POSE = [
    (255, 80, 80), (80, 255, 80), (80, 80, 255), (255, 255, 80), (255, 80, 255),
]
CONF_KP = 0.3   # minimum keypoint confidence to draw

def draw_skeleton(canvas, pose, colour):
    kps = pose.keypoints   # (17, 3): x, y, conf
    for kp_idx in range(17):
        x, y, c = kps[kp_idx]
        if c >= CONF_KP:
            cv2.circle(canvas, (int(x), int(y)), 4, colour, -1)
    for (a, b) in SKELETON:
        xa, ya, ca = kps[a]; xb, yb, cb = kps[b]
        if ca >= CONF_KP and cb >= CONF_KP:
            cv2.line(canvas, (int(xa), int(ya)), (int(xb), int(yb)), colour, 2)

SAMPLE_EVERY = max(1, len(frames) // 9)
sample_idx   = list(range(0, len(frames), SAMPLE_EVERY))[:9]

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, fi in zip(axes.flat, sample_idx):
    canvas = frames[fi].copy()
    for j, pose in enumerate(all_poses[fi]):
        draw_skeleton(canvas, pose, PALETTE_POSE[pose.track_id % len(PALETTE_POSE)])
    ax.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    ax.set_title(f"t={timestamps[fi]:.1f}s  poses={len(all_poses[fi])}", fontsize=9)
    ax.axis('off')

plt.suptitle('RTMPose-m — Skeleton Overlay', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Keypoint confidence distribution ─────────────────────────────────────────
KP_NAMES = [
    'nose','left_eye','right_eye','left_ear','right_ear',
    'left_shoulder','right_shoulder','left_elbow','right_elbow',
    'left_wrist','right_wrist','left_hip','right_hip',
    'left_knee','right_knee','left_ankle','right_ankle',
]

all_kp_confs = np.array([
    p.keypoints[:, 2]
    for frame_poses in all_poses for p in frame_poses
]) if any(all_poses) else np.zeros((0, 17))

if len(all_kp_confs) > 0:
    mean_conf = all_kp_confs.mean(axis=0)
    fig, ax = plt.subplots(figsize=(14, 4))
    bars = ax.bar(KP_NAMES, mean_conf, color='steelblue', edgecolor='white')
    ax.axhline(CONF_KP, color='red', linestyle='--', label=f'threshold={CONF_KP}')
    ax.set_ylabel('Mean confidence'); ax.set_title('Mean keypoint confidence across all poses')
    ax.set_xticklabels(KP_NAMES, rotation=45, ha='right', fontsize=8)
    ax.legend(); plt.tight_layout(); plt.show()
    print(f"Total poses: {len(all_kp_confs)}")
    print(f"Mean pose confidence: {all_kp_confs.mean():.3f}")
else:
    print("No poses detected.")